# Milestone 1 lab, notebook 4 of 7: Transform, then load

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and the source | Read the question. Meet the order system. Look at one order |
| 2. Read every order | Follow the pages of the API to the end |
| 3. The answer by hand | Work out the answer with plain Python. Meet the time zone trap |
| **4. Transform, then load** (you are here) | Put the orders in a warehouse table. Ask it one SQL question |
| 5. Every morning | Run it again the next day. Watch an old number change |
| 6. Why batch, and where it breaks | Measure how late the report is. Then watch the load fail |
| 7. The production version | The same logic as a tested program |

In notebook 3 the answer lived in Python's memory. Close the notebook and it is gone. Nobody else can ask it a question.

Now the orders go into a **warehouse**: a database built for analysis. Then one SQL query answers the manager.

In [1]:
import duckdb
import pandas as pd
from lab_helpers import LAB_WAREHOUSE, get_all_orders, load_result, money, set_morning, show_doc

set_morning("2026-06-08")
orders, calls = get_all_orders()
print(f"{len(orders)} orders in {calls} calls.")

It is now Monday 2026-06-08 07:00:00 in Arden.
226 orders in 10 calls.


## Step 1: The plan

In [2]:
show_doc("HLD.md", "4. What each part is responsible for")

**From `docs/HLD.md`**

> ## 4. What each part is responsible for
> 
> | Part | Responsibility | Does not do |
> |---|---|---|
> | **Extract** | Follow the pages to the end. Count the calls. Stop with a clear error if the API refuses | Change any value |
> | **Transform** | Keep the columns the report needs. Turn `placed_at` into a business day | Talk to the API or the warehouse |
> | **Load** | Replace the whole `orders` table in one step. Keep the last table as `orders_previous` | Decide what a row means |
> | **Report** | Answer the four parts of the question | Change any table |
> 
> Each part can be tested alone, because each has one job.

You built **Extract** in notebook 2. This notebook builds the other three.

## Step 2: Transform

The orders arrive as JSON. A table needs flat rows and columns. **pandas** is a Python library for tables. It turns the list into one in a single line.

In [3]:
raw = pd.DataFrame(orders)

print(f"{len(raw)} rows, {len(raw.columns)} columns")
raw.head(3)

226 rows, 12 columns


,order_id,customer_id,city,status,placed_at,updated_at,delivered_at,cancelled_at,total_cents,refunded_cents,refunded_at,items
0,1001,1227,NGT,delivered,2026-06-01T12:10:06Z,2026-06-01T14:45:06Z,2026-06-01T14:45:06Z,NaN,3276,0,None,"[{'line_no': 1, 'product_id': 'P-006', 'produc..."
1,1002,1001,LKS,delivered,2026-06-01T13:17:46Z,2026-06-01T15:25:46Z,2026-06-01T15:25:46Z,NaN,2512,0,None,"[{'line_no': 1, 'product_id': 'P-008', 'produc..."
2,1004,1010,RVB,delivered,2026-06-01T13:46:54Z,2026-06-01T15:42:54Z,2026-06-01T15:42:54Z,NaN,2721,0,None,"[{'line_no': 1, 'product_id': 'P-028', 'produc..."


Two things are not ready for a report:

1. The times are text, in UTC. The report needs a **business day**.
2. The `items` column holds a whole list in every cell. A table cell should hold one value.

Fix the day first. It is the rule from notebook 3, now on a whole column at once.

In [4]:
raw["placed_at"] = pd.to_datetime(raw["placed_at"], utc=True)
raw["updated_at"] = pd.to_datetime(raw["updated_at"], utc=True)

raw["order_date"] = raw["placed_at"].dt.tz_convert("America/Chicago").dt.date

raw.loc[raw["order_id"] == 1196, ["order_id", "placed_at", "order_date"]]

,order_id,placed_at,order_date
190,1196,2026-06-07 04:58:12+00:00,2026-06-06


Order 1196, the two-minutes-to-midnight order, lands on Saturday 6 June. Correct.

Now the items. This question needs only a count of them. Then keep the columns the report needs, and drop the rest.

In [5]:
raw["item_count"] = raw["items"].apply(
    lambda items: sum(1 for item in items if item["item_status"] == "ok")
)

clean = raw[
    [
        "order_id",
        "customer_id",
        "city",
        "status",
        "placed_at",
        "order_date",
        "item_count",
        "total_cents",
        "refunded_cents",
        "updated_at",
    ]
]

print(f"{len(clean)} rows, {len(clean.columns)} columns. The grain is one row per order.")
clean.tail(3)

226 rows, 10 columns. The grain is one row per order.


,order_id,customer_id,city,status,placed_at,order_date,item_count,total_cents,refunded_cents,updated_at
223,1224,1177,NGT,placed,2026-06-08 03:34:57+00:00,2026-06-07,5,7018,0,2026-06-08 03:34:57+00:00
224,1225,1063,NGT,placed,2026-06-08 03:38:57+00:00,2026-06-07,4,4135,0,2026-06-08 03:38:57+00:00
225,1226,1194,NGT,placed,2026-06-08 03:48:53+00:00,2026-06-07,5,2283,0,2026-06-08 03:48:53+00:00


## Step 3: Load

The warehouse here is **DuckDB**. It is a database that lives in one file, so there is nothing to install or start. Milestone 2 moves to Postgres.

Loading is one statement: make a table from the rows.

In [6]:
LAB_WAREHOUSE.unlink(missing_ok=True)  # start from an empty warehouse
con = duckdb.connect(str(LAB_WAREHOUSE))
con.execute("SET TimeZone = 'UTC'")

con.execute("CREATE TABLE orders AS SELECT * FROM clean")

con.sql(
    "SELECT count(*) AS rows_in_warehouse, min(order_date) AS first_day, max(order_date) AS last_day FROM orders"
).df()

,rows_in_warehouse,first_day,last_day
0,226,2026-06-01,2026-06-07


The orders are now in a file on your disk. They will still be there tomorrow.

## Step 4: Ask the warehouse

Here is the report as one SQL query. One row per business day, per city.

In [7]:
REPORT_SQL = """
SELECT
    order_date,
    city,
    count(*) FILTER (WHERE status <> 'cancelled')                 AS orders,
    count(*) FILTER (WHERE status = 'cancelled')                  AS cancelled_orders,
    coalesce(sum(total_cents - refunded_cents)
             FILTER (WHERE status <> 'cancelled'), 0)::BIGINT     AS revenue_cents
FROM orders
GROUP BY order_date, city
ORDER BY order_date, city
"""

report = con.sql(REPORT_SQL).df()
report[report["order_date"].astype(str) >= "2026-06-06"]

,order_date,city,orders,cancelled_orders,revenue_cents
15,2026-06-06,LKS,9,0,20078
16,2026-06-06,NGT,17,0,39599
17,2026-06-06,RVB,7,0,18994
18,2026-06-07,LKS,6,0,14498
19,2026-06-07,NGT,10,0,35526
20,2026-06-07,RVB,14,0,30687


`FILTER (WHERE status <> 'cancelled')` is the cancelled rule from notebook 3. `GROUP BY order_date, city` sets the grain of the result: one row per day, per city.

Now the manager's two days, with the cities added together.

In [8]:
days = con.sql(f"""
    SELECT order_date, sum(orders)::BIGINT AS orders, sum(revenue_cents)::BIGINT AS revenue_cents
    FROM ({REPORT_SQL})
    WHERE order_date IN ('2026-06-07', '2026-06-06')
    GROUP BY order_date
    ORDER BY order_date DESC
""").df()
days

,order_date,orders,revenue_cents
0,2026-06-07,30,80711
1,2026-06-06,33,78671


## Step 5: Hold it up against the answer key

This is the habit of the whole project. A new version is right only when it equals the key.

In [9]:
key = load_result("answer_key")

for label in ["yesterday", "day_before"]:
    expected = key[label]
    row = days[days["order_date"].astype(str) == expected["order_date"]].iloc[0]
    same = (int(row["orders"]), int(row["revenue_cents"])) == (
        expected["orders"],
        expected["revenue_cents"],
    )
    print(
        f"{expected['order_date']}   SQL: {int(row['orders'])} orders {money(int(row['revenue_cents']))}"
        f"   key: {expected['orders']} orders {money(expected['revenue_cents'])}   {'✅ same' if same else '❌ DIFFERENT'}"
    )

2026-06-07   SQL: 30 orders $807.11   key: 30 orders $807.11   ✅ same
2026-06-06   SQL: 33 orders $786.71   key: 33 orders $786.71   ✅ same


## Step 6: A new word, ETL

Look at the order you just worked in:

1. **E**xtract: read the orders from the source.
2. **T**ransform: change them in Python.
3. **L**oad: write the result to the warehouse.

That order is called **ETL**.

> "ETL transforms the data and then loads only the result. ELT loads the raw data and then transforms it inside the warehouse."

The quote is from the D8Loop chapter [ETL vs ELT](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/etl-vs-elt). You will build ELT in milestone 4 and compare the two on the same data.

### Feel the cost of ETL

ETL loads **only the result**. Count what arrived, and what landed.

In [10]:
arrived = sum(len(order["items"]) for order in orders)
print(f"The API sent      {len(orders)} orders with {arrived} item lines.")
print(
    f"The warehouse has {con.sql('SELECT count(*) FROM orders').fetchall()[0][0]} orders with 0 item lines."
)

The API sent      226 orders with 759 item lines.
The warehouse has 226 orders with 0 item lines.


Next week a city manager asks a new question: *which product sold the most units on Sunday?* Ask the warehouse.

In [11]:
try:
    con.sql(
        "SELECT product_name, sum(quantity) FROM orders WHERE order_date = '2026-06-07' GROUP BY 1"
    )
except duckdb.Error as error:
    print("The warehouse says:")
    print(" ", str(error).splitlines()[0])

The warehouse says:
  Binder Error: Referenced column "product_name" not found in FROM clause!


The warehouse cannot answer. The product names were dropped before the load.

To answer, you must change the Python and **read the source again**. And the source keeps changing. If you ask next week, you get next week's orders, not the ones behind today's report.

Remember this. It is the reason milestone 2 keeps a raw copy of everything that arrives.

## Step 7: The five stages

Every pipeline has the same five stages. Here is where this notebook stands.

In [12]:
show_doc("HLD.md", "2. The five stages, and where this milestone stands")

**From `docs/HLD.md`**

> ## 2. The five stages, and where this milestone stands
> 
> Every pipeline has the same five stages. This milestone builds a first version of four.
> 
> | Stage | Its job | In this milestone | State |
> |---|---|---|---|
> | **Ingest** | Copy the data in | Read every page of `GET /orders` | ✅ Built, the simple way |
> | **Store** | Keep every layer of the data | One clean table. The raw copy is thrown away | ⚠️ Half built |
> | **Transform** | Clean the data and add it up | pandas, then one SQL query | ✅ Built |
> | **Serve** | Give the result to its readers | Printed report | ✅ Built |
> | **Observe** | Check every other stage | Nothing checks anything | ❌ Not yet |
> 
> The same map is on the milestone's [flow page](../milestone_1_flow.html). Each later milestone fills in more of it.

In [13]:
con.close()

The stage names are from the D8Loop chapter [The Data Pipeline Lifecycle](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/the-data-pipeline-lifecycle).

## What you now know

- pandas turned 226 JSON orders into a table. One line set the business day for every row.
- The warehouse is a database built for analysis. Here it is one DuckDB file.
- One SQL query gave the same answer as the hand-worked key.
- ETL transforms before the load, so only the result lands. A new question means reading the source again.

**Say it in an interview:** "I started with ETL: transform in pandas, then load only the columns the report needed. It worked, but when someone asked a question about items, the warehouse could not answer, because the items were dropped before the load. That is why I moved to loading the raw data first."

Next: [notebook 5](05_every_morning.ipynb) runs all of this again the next morning.